# Checkpoint 04 - visualização em tempo real

Mantemos a aquisição, a validação e os buffers construídos até o **Checkpoint 03**.

A novidade é transformar a janela armazenada nos buffers em gráficos atualizados em tempo real, mantendo o eixo horizontal baseado no relógio da aquisição.

## 1. Configuração

In [ ]:
# CONFIGURAÇÃO DA OFICINA
# Use a mesma URL do Firebase Realtime Database fornecida para a oficina.
FIREBASE_URL = "https://oficina-python-saude-teste-default-rtdb.firebaseio.com"  # Endereço base do Realtime Database usado na oficina.
STREAM_PATH = "oficina/live"  # Caminho do nó que contém sempre o pacote mais recente.

# Parâmetros da oficina
FS = 250  # Frequência de amostragem dos sinais: 250 amostras por segundo.
AMOSTRAS_POR_PACOTE = 50  # Quantidade esperada de amostras em cada pacote publicado.
JANELA_S = 8  # Duração, em segundos, da janela de sinal mantida no buffer.

## 2. Reutilização da comunicação e validação

Nada muda na forma de receber os pacotes: continuamos usando REST/SSE e `validar_pacote()`.

In [ ]:
import json                         # Converte as mensagens JSON recebidas pelo SSE.
import time                         # Controla a duração da captura e o intervalo entre atualizações do gráfico.
from collections import deque       # Mantém apenas a janela mais recente dos sinais.
import numpy as np                  # Organiza os vetores numéricos e os instantes do eixo do tempo.
import requests                     # Recebe o fluxo do Firebase por HTTP/SSE.

# Função: Monta a URL REST completa do nó do Firebase, evitando barras duplicadas.
def firebase_node_url(firebase_url, node=STREAM_PATH):
    base = firebase_url.rstrip("/")  # Remove uma barra final para evitar "//" ao montar a URL.
    node = node.strip("/")  # Remove barras no início/fim do caminho antes de juntá-lo à URL.
    return f"{base}/{node}.json"  # O sufixo .json permite acessar esse nó pela API REST do Firebase.

# Função: Faz uma leitura única do pacote atualmente armazenado no nó da oficina.
def ler_pacote_atual(firebase_url=FIREBASE_URL, node=STREAM_PATH):
    r = requests.get(firebase_node_url(firebase_url, node), timeout=10)  # Faz um GET e espera no máximo 10 s pela resposta.
    r.raise_for_status()  # Interrompe com erro se o servidor responder com código HTTP de falha.
    return r.json()  # Converte a resposta JSON do Firebase diretamente em um objeto Python.


# Função: Separa o fluxo SSE em eventos completos, identificando os campos event e data.
def iterar_sse(response):
    """Interpreta eventos Server-Sent Events retornados pelo Firebase REST."""
    evento = None
    dados = []

    for raw in response.iter_lines(decode_unicode=True):  # Lê a resposta contínua linha a linha, já convertida para texto.
        if raw is None:
            continue

        linha = raw.strip()  # Remove espaços/quebras de linha antes de interpretar o campo SSE.

        if linha == "":  # No SSE, uma linha vazia indica o fim de um evento.
            if evento is not None and dados:
                yield evento, "\n".join(dados)
            evento = None
            dados = []
            continue

        if linha.startswith(":"):  # Linhas iniciadas por ":" são comentários/keep-alive do protocolo SSE.
            continue

        if linha.startswith("event:"):
            evento = linha.split(":", 1)[1].strip()  # Extrai apenas o valor que vem depois de "event:".
        elif linha.startswith("data:"):
            dados.append(linha.split(":", 1)[1].lstrip())  # Guarda apenas o conteúdo que vem depois de "data:".

# Função: Mantém a conexão SSE aberta e entrega, um a um, os pacotes válidos recebidos.
def pacotes_firebase(firebase_url=FIREBASE_URL, node=STREAM_PATH, duracao_s=20):
    """Gera pacotes recebidos pelo stream SSE do Firebase."""
    url = firebase_node_url(firebase_url, node)  # Obtém a URL REST completa do nó que será acompanhado.
    headers = {"Accept": "text/event-stream"}  # Informa ao Firebase que queremos receber Server-Sent Events.
    inicio = time.time()  # Marca o instante inicial para limitar a duração da captura.

    with requests.get(url, headers=headers, stream=True, timeout=(10, 35)) as response:
        response.raise_for_status()  # Garante que a conexão foi aceita antes de processar o fluxo.
        for evento, texto in iterar_sse(response):
            if time.time() - inicio >= duracao_s:
                return
            if evento not in ("put", "patch"):
                continue

            envelope = json.loads(texto)  # Transforma o texto JSON do evento SSE em dicionário Python.
            data = envelope.get("data")  # No SSE do Firebase, o conteúdo alterado vem dentro da chave data.
            if isinstance(data, dict) and "seq" in data:
                yield data  # Entrega o pacote ao laço for sem encerrar a conexão SSE.

# Função: Confere os campos e tamanhos do pacote e converte os sinais para vetores NumPy.
def validar_pacote(pacote):
    campos = (
        "session_id", "seq", "sample_start", "fs", "n",
        "ecg", "ppg_ir", "ppg_red"
    )
    faltando = [k for k in campos if k not in pacote]  # Lista quais campos obrigatórios não chegaram no pacote.
    if faltando:
        raise ValueError(f"campos ausentes: {faltando}")

    n = int(pacote["n"])  # Converte o tamanho informado para inteiro antes de comparar com os vetores.
    ecg = np.asarray(pacote["ecg"], dtype=float)  # Converte a lista de ECG recebida em vetor numérico.
    ir = np.asarray(pacote["ppg_ir"], dtype=float)  # Converte o canal PPG infravermelho em vetor numérico.
    red = np.asarray(pacote["ppg_red"], dtype=float)  # Converte o canal PPG vermelho em vetor numérico.

    if not (len(ecg) == len(ir) == len(red) == n):
        raise ValueError("vetores com tamanhos incompatíveis")

    return {
        "session_id": str(pacote["session_id"]),
        "seq": int(pacote["seq"]),
        "sample_start": int(pacote["sample_start"]),
        "fs": float(pacote["fs"]),
        "n": n,
        "timestamp_pub": float(pacote.get("timestamp_pub", 0.0)),
        "ecg": ecg,
        "ppg_ir": ir,
        "ppg_red": red,
    }

## 3. Reutilização da lógica de buffer do Checkpoint 03

A mesma ideia de janela temporal e de `NaN` para lacunas permanece.
Neste notebook a função recebe o nome `append_with_gaps()`, deixando explícito que ela acrescenta os pacotes preservando as lacunas.

In [ ]:
sample_buf = deque()  # Deque permite acrescentar e remover amostras das extremidades com baixo custo.
ecg_buf = deque()  # Deque permite acrescentar e remover amostras das extremidades com baixo custo.
ir_buf = deque()  # Deque permite acrescentar e remover amostras das extremidades com baixo custo.
red_buf = deque()  # Deque permite acrescentar e remover amostras das extremidades com baixo custo.
active_session = None
last_sample_end = None
last_plot = 0.0

# Função: Acrescenta o pacote aos buffers e insere NaN quando há uma lacuna de amostras.
def append_with_gaps(p):
    global active_session, last_sample_end  # Permite que a função atualize o estado da sessão mantido fora dela.
    if active_session != p["session_id"]:
        active_session = p["session_id"]
        last_sample_end = None
        sample_buf.clear()  # Reinicia os índices ao mudar de sessão.
        ecg_buf.clear()  # Reinicia o ECG.
        ir_buf.clear()  # Reinicia o PPG infravermelho.
        red_buf.clear()  # Reinicia o PPG vermelho.

    if last_sample_end is not None:
        esperado = last_sample_end + 1
        for sample_idx in range(esperado, p["sample_start"]):
            sample_buf.append(sample_idx)
            ecg_buf.append(np.nan)  # NaN faz o gráfico interromper a linha do ECG.
            ir_buf.append(np.nan)  # Preserva a mesma lacuna no PPG IR.
            red_buf.append(np.nan)  # Preserva a mesma lacuna no PPG Red.

    for i in range(p["n"]):
        sample_buf.append(p["sample_start"] + i)
        ecg_buf.append(float(p["ecg"][i]))
        ir_buf.append(float(p["ppg_ir"][i]))
        red_buf.append(float(p["ppg_red"][i]))

    last_sample_end = p["sample_start"] + p["n"] - 1  # Guarda o índice da última amostra válida deste pacote.
    minimo = last_sample_end - int(JANELA_S * p["fs"])  # Calcula o índice mínimo que ainda pertence à janela temporal.
    while sample_buf and sample_buf[0] < minimo:
        sample_buf.popleft()  # Remove o índice mais antigo.
        ecg_buf.popleft()  # Remove a amostra de ECG correspondente.
        ir_buf.popleft()  # Remove a amostra PPG IR correspondente.
        red_buf.popleft()  # Remove a amostra PPG Red correspondente.

## 4. Novidade deste checkpoint — visualização em tempo real

Agora entram `matplotlib` e os recursos de atualização do Colab.

A função `mostrar_graficos()` apenas **visualiza os buffers já construídos**; ela não altera o tempo do sinal nem os dados recebidos.

In [ ]:
import matplotlib.pyplot as plt     # Desenha os gráficos de ECG e PPG.
from IPython.display import clear_output, display  # Atualiza o gráfico no mesmo espaço de saída do Colab.

# Função: Desenha ECG e PPG usando o relógio da aquisição, e não o instante de chegada pela rede.
def mostrar_graficos(p):
    latest = sample_buf[-1]  # Usa a amostra mais recente como referência de tempo zero.
    tx = np.asarray([(s - latest) / p["fs"] for s in sample_buf])  # Converte índices de amostra em segundos relativos ao ponto mais recente.

    clear_output(wait=True)  # Limpa a saída anterior para atualizar o gráfico no mesmo lugar.
    fig, ax = plt.subplots(2, 1, figsize=(11, 6), sharex=True)

    ax[0].plot(tx, list(ecg_buf))
    ax[0].set_title("ECG - tempo definido pela aquisição")
    ax[0].set_ylabel("amplitude")
    ax[0].grid(True, alpha=.25)

    ax[1].plot(tx, list(ir_buf), label="PPG IR")
    ax[1].plot(tx, list(red_buf), label="PPG Red")
    ax[1].set_title("PPG - tempo definido pela aquisição")
    ax[1].set_xlabel("tempo relativo ao sinal (s)")
    ax[1].set_ylabel("contagens")
    ax[1].legend()
    ax[1].grid(True, alpha=.25)

    fig.tight_layout()
    display(fig)  # Exibe explicitamente a figura atualizada no notebook.
    plt.close(fig)  # Fecha a figura depois de exibi-la para não acumular objetos na memória.

## 5. Executar

O laço continua recebendo e armazenando os pacotes como antes.  
A única etapa acrescentada é chamar `mostrar_graficos()` periodicamente.

In [ ]:
for pacote in pacotes_firebase(duracao_s=35):
    p = validar_pacote(pacote)
    append_with_gaps(p)

    agora = time.time()  # Obtém o instante atual para controlar a atualização da tela/rede.
    if agora - last_plot < 0.8 or len(sample_buf) < 10:  # Evita redesenhar rápido demais ou com poucas amostras.
        continue
    last_plot = agora

    mostrar_graficos(p)  # Usa a nova função de visualização definida neste checkpoint.

print("CHECKPOINT 4 OK")